Introduction
============
**Topic**: Introduction to NetworkX

This notebook is a part of a collection of notebooks and includes basic information and examples to implement graphs using NetworkX and apply them in path planning algorithms.

Version | Author
------------ | -------------
0.2 | Björn Hein

License is based on Creative Commons: Attribution-NonCommercial 4.0 International (CC BY-NC 4.0) (pls. check: http://creativecommons.org/licenses/by-nc/4.0/)

Content:

* [Graph handling](#Graph-handling)
* [Drawing graphs](#Drawing-graphs)
* [Lecture examples](#Lecture-examples)
* [Layouting](#Layouting)
* [Planning-related graph examples](#Planning-related-graph-examples)
* [Conclusion](#Conclusion)
* [Exercise](#Exercise)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

print(nx.__version__)

# Graph handling
## Creating a graph

`nx.Graph()` creates an empty undirected graph (directed graphs: `nx.DiGraph()`, see below).

In [ ]:
G = nx.Graph()

## Adding Nodes

A node can be any hashable Python object: numbers, strings, tuples, ...

In [ ]:
G.add_node(1)                   # a single node
G.add_nodes_from([2, 3, "A"])   # several nodes at once
print(G)

## Adding Edges

Adding an edge also adds its nodes, if they do not exist yet.

In [ ]:
G.add_edge(1, 2)
e = (2, 3)
G.add_edge(*e)                  # edge given as tuple
G.add_edge("Bob", "Marie")      # "Bob" and "Marie" are added as nodes automatically
G.add_edge("Bob", "A")

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print("Nodes:", list(G.nodes()))
print("Edges:", list(G.edges()))

## Neighbors and degree

Planning algorithms permanently need to know which nodes are connected to a given node.

In [ ]:
print("Neighbors of 2:", list(G.neighbors(2)))
print("Degree of 2:", G.degree(2))   # number of edges at node 2

## Node attributes

Every node has a dictionary for arbitrary attributes. `G.nodes(data=True)` lists the nodes together with their attributes.

In [ ]:
print(G.nodes(data=True))   # no attributes yet

G.nodes[1]["color"] = "red"
G.nodes[1]["time"] = 4.3
G.nodes["A"]["Amazing"] = "Parameter"

print(G.nodes[1])
print(G.nodes["A"])
print(G.nodes(data=True))

## Edge attributes

The same applies to edges: `G[u][v]` is the attribute dictionary of the edge between `u` and `v`.

In [ ]:
G["Bob"]["Marie"]["weight"] = 100.3
G.add_edge(1, 2, weight=2.5)   # for an existing edge only the attribute is set

print(G["Bob"]["Marie"])
print(G.edges(data=True))

## Analyzing

`nx.connected_components` returns the sets of nodes, which are connected to each other.

In [ ]:
for component in nx.connected_components(G):
    print(component)

Now the two components are connected. Note: the integer `1` and the string `'1'` are two different nodes.

In [ ]:
G.add_edge("A", 1)
G.add_edge(1, "1")

for component in nx.connected_components(G):
    print(component)
print("Connected:", nx.is_connected(G))

# Drawing graphs

`nx.draw` draws a graph with matplotlib. Without given positions a layout algorithm places the nodes; with a fixed `seed` the picture is the same in every run. The labels are created with `repr`, so `1` and `'1'` can be distinguished.

In [ ]:
# colors used in this notebook (same convention as in the other notebooks: start green, goal red)
NODE_COLOR = "#b7d3f6"
START_COLOR = "#0ca30c"
GOAL_COLOR = "#d03b3b"
PATH_COLOR = "#2a78d6"

fig, axes = plt.subplots()
labels = {node: repr(node) for node in G}
# weight=None: ignore the edge weights, otherwise the weight 100.3 pulls "Bob" and "Marie" on top of each other
layout = nx.spring_layout(G, seed=19, weight=None)
nx.draw(G, layout, ax=axes, labels=labels, node_color=NODE_COLOR, node_size=900)

# Lecture examples
Following examples are according to the slides provided in the lecture.

## Initializing graphs
### Undirected graph

In [ ]:
G_undir = nx.Graph()
G_undir.add_nodes_from(["A", "B", "C", "D"])
G_undir.add_edges_from([("A", "B"), ("B", "C"), ("C", "D"), ("D", "B")])

Node colors can be given as a list with one color per node (in the order of `G.nodes()`). It is more robust to derive this list from a dictionary. The positions of the nodes are computed once and used for all following drawings of these graphs.

In [ ]:
color_map = {"A": START_COLOR, "D": GOAL_COLOR}   # start and goal of the path search below
colors = [color_map.get(node, NODE_COLOR) for node in G_undir]
print(colors)

pos = nx.spiral_layout(G_undir)   # dictionary: node -> (x, y)
print(pos)

fig, axes = plt.subplots()
nx.draw(G_undir, pos, ax=axes, with_labels=True, node_color=colors, node_size=600)
# Remark: ax=axes can be omitted if you just want to draw in the last generated figure (e.g. by plt.subplots() or plt.figure(), etc)
# It gives you the possibility to draw in a specified figure, even when plotted in the notebook before.

### Directed graph

In [ ]:
G_dir = nx.DiGraph()
G_dir.add_nodes_from(["A", "B", "C", "D"])
G_dir.add_edges_from([("A", "B"), ("B", "C"), ("C", "D"), ("D", "B")])

fig, axes = plt.subplots()
nx.draw(G_dir, pos, ax=axes, with_labels=True, node_color=colors, node_size=600)

## Add attribute to edges
### Undirected graph

In [ ]:
# set weights of the edges
G_undir["A"]["B"]["weight"] = 3
G_undir["B"]["C"]["weight"] = 2
G_undir["C"]["D"]["weight"] = 4
G_undir["D"]["B"]["weight"] = 10

fig, axes = plt.subplots()
nx.draw(G_undir, pos, ax=axes, with_labels=True, node_color=colors, node_size=600)

# get a specific edge attribute and draw it
edge_attributes = nx.get_edge_attributes(G_undir, "weight")
nx.draw_networkx_edge_labels(G_undir, pos, ax=axes, edge_labels=edge_attributes)
plt.show()

### Directed Graph

In a directed graph B→D and D→B are two different edges, each with its own weight. The edges are drawn curved, so that both are visible.

In [ ]:
G_dir["A"]["B"]["weight"] = 2
G_dir["B"]["C"]["weight"] = 3
G_dir["C"]["D"]["weight"] = 8
G_dir["D"]["B"]["weight"] = 7
G_dir.add_edge("B", "D", weight=4)   # new edge B -> D, created together with its weight

curved = "arc3,rad=0.15"   # bend the edges, so B -> D and D -> B do not overlap

fig, axes = plt.subplots()
nx.draw(G_dir, pos, ax=axes, with_labels=True, node_color=colors, node_size=600, connectionstyle=curved)
edge_attributes = nx.get_edge_attributes(G_dir, "weight")
nx.draw_networkx_edge_labels(G_dir, pos, ax=axes, edge_labels=edge_attributes, connectionstyle=curved)
plt.show()

## Finding paths
https://networkx.org/documentation/stable/reference/algorithms/shortest_paths.html

### Undirected Graph

In [ ]:
shortest_path = nx.shortest_path(G_undir, source="A", target="D", weight="weight")
print(shortest_path)

In [ ]:
fig, axes = plt.subplots()
nx.draw(G_undir, pos, ax=axes, with_labels=True, node_color=colors, node_size=600)

# draw the solution path on top, then the edge weights
edgeList = list(zip(shortest_path[:-1], shortest_path[1:]))   # [("A", "B"), ("B", "C"), ...]
nx.draw_networkx_edges(G_undir, pos, ax=axes, edgelist=edgeList, edge_color=PATH_COLOR, width=4.0)
nx.draw_networkx_edge_labels(G_undir, pos, ax=axes, edge_labels=nx.get_edge_attributes(G_undir, "weight"))
plt.show()

In [ ]:
# costs of the path = sum of the edge weights
costs_undir = sum(G_undir[u][v]["weight"] for u, v in edgeList)
print(costs_undir)
print(nx.path_weight(G_undir, shortest_path, weight="weight"))   # the same with a NetworkX function

### Directed Graph

The path example of the slides does not use the edge B→D. Instead of removing it from `G_dir`, the edge is removed from a copy, so `G_dir` stays unchanged for the following sections.

In [ ]:
G_dir_path = G_dir.copy()
G_dir_path.remove_edge("B", "D")

shortest_path = nx.shortest_path(G_dir_path, source="A", target="D", weight="weight")
print(shortest_path)

In [ ]:
fig, axes = plt.subplots()
nx.draw(G_dir_path, pos, ax=axes, with_labels=True, node_color=colors, node_size=600, connectionstyle=curved)

edgeList = list(zip(shortest_path[:-1], shortest_path[1:]))
nx.draw_networkx_edges(G_dir_path, pos, ax=axes, edgelist=edgeList, edge_color=PATH_COLOR, width=4.0,
                       node_size=600, connectionstyle=curved)
nx.draw_networkx_edge_labels(G_dir_path, pos, ax=axes, edge_labels=nx.get_edge_attributes(G_dir_path, "weight"),
                             connectionstyle=curved)
plt.show()

In [ ]:
costs_dir = nx.path_weight(G_dir_path, shortest_path, weight="weight")
print(costs_dir)

## Further tests

In [ ]:
print("Connected:", nx.is_connected(G_undir))
print("Cycle basis:", nx.cycle_basis(G_undir))
for cycle in nx.simple_cycles(G_dir_path):   # directed graph without B -> D, as in the path example
    print("Cycle:", cycle)

## Representation

Adjacency matrix of the directed graph (with the edge B→D as in the slides). Rows and columns follow the given node order, the entries are the edge weights.

Check https://networkx.org/documentation/stable/reference/generated/networkx.linalg.graphmatrix.adjacency_matrix.html !

In [ ]:
node_order = sorted(G_dir.nodes())
print(node_order)

adj_matrix = nx.adjacency_matrix(G_dir, nodelist=node_order)
adj_matrix.toarray()

# Layouting

In the planning environment it will be important to place the node at certain positions. In the following some hints are given how to use attributes of nodes in NetworkX to do so.

The attribute is called `mypos` here to show that NetworkX has no dedicated position attribute. The planners of this course store the position of a node in the attribute `pos`.

In [ ]:
G_graph = nx.Graph()
G_graph.add_node("A", mypos=[0, 1])
G_graph.add_node("B", mypos=[10, 1])
G_graph.add_node("C", mypos=[1, 10])
G_graph.add_node("D", mypos=[1, 8])
G_graph.add_edge("A", "B")
G_graph.add_edge("B", "C")

for node in G_graph.nodes(data=True):
    print(node)

In [ ]:
pos_dict = nx.get_node_attributes(G_graph, "mypos")
print(pos_dict)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
nx.draw(G_graph, pos=pos_dict, ax=axes[0], with_labels=True, node_color=NODE_COLOR)

# nodes and edges can also be drawn separately, e.g. to style the edges
nx.draw_networkx_nodes(G_graph, pos_dict, ax=axes[1], node_color=NODE_COLOR)
nx.draw_networkx_edges(G_graph, pos_dict, ax=axes[1], edge_color=PATH_COLOR, width=3.0)
plt.show()

# Planning-related graph examples

The examples above show the basic NetworkX API. For path planning, the graph usually represents a roadmap: nodes store configurations or positions and edges store feasible local connections. Edge weights often represent geometric distance or traversal cost.

## Weighted and unweighted shortest paths

Without a `weight` parameter, NetworkX minimizes the number of edges. With `weight="weight"`, it minimizes the sum of edge weights.

In [ ]:
G_cost = nx.Graph()
G_cost.add_edge("start", "A", weight=1)
G_cost.add_edge("A", "goal", weight=10)
G_cost.add_edge("start", "B", weight=3)
G_cost.add_edge("B", "C", weight=3)
G_cost.add_edge("C", "goal", weight=3)

print("Unweighted shortest path:", nx.shortest_path(G_cost, "start", "goal"))
print("Weighted shortest path:", nx.shortest_path(G_cost, "start", "goal", weight="weight"))
print("Weighted path length:", nx.shortest_path_length(G_cost, "start", "goal", weight="weight"))

In [ ]:
pos_cost = {
    "start": (0, 0),
    "A": (1, 1),
    "goal": (4, 0),
    "B": (1, -1),
    "C": (3, -1),
}
weighted_path = nx.shortest_path(G_cost, "start", "goal", weight="weight")
path_edges = list(zip(weighted_path[:-1], weighted_path[1:]))
node_colors = [{"start": START_COLOR, "goal": GOAL_COLOR}.get(node, NODE_COLOR) for node in G_cost]

fig, axes = plt.subplots()
nx.draw(G_cost, pos_cost, with_labels=True, node_color=node_colors, node_size=900, ax=axes)
nx.draw_networkx_edges(G_cost, pos_cost, edgelist=path_edges, edge_color=PATH_COLOR, width=4, ax=axes)
nx.draw_networkx_edge_labels(
    G_cost,
    pos_cost,
    edge_labels=nx.get_edge_attributes(G_cost, "weight"),
    ax=axes,
)
plt.show()

## Geometric roadmap graph

In a roadmap graph, every node can store a position. Edges are added when two nodes are close enough. The edge weight is the Euclidean distance between the positions.

In [ ]:
import math
import random

def build_roadmap(num_nodes, connection_radius, seed):
    """Nodes at random positions in the unit square, edges between nodes closer than connection_radius."""
    rng = random.Random(seed)
    graph = nx.Graph()
    for node_id in range(num_nodes):
        graph.add_node(node_id, pos=(rng.random(), rng.random()))

    for node_a in graph.nodes:
        for node_b in graph.nodes:
            if node_a >= node_b:
                continue
            distance = math.dist(graph.nodes[node_a]["pos"], graph.nodes[node_b]["pos"])
            if distance <= connection_radius:
                graph.add_edge(node_a, node_b, weight=distance)
    return graph

num_nodes = 25
G_roadmap = build_roadmap(num_nodes, connection_radius=0.32, seed=7)

print(G_roadmap)
print("Connected:", nx.is_connected(G_roadmap))
print("Number of connected components:", nx.number_connected_components(G_roadmap))

In [ ]:
pos_roadmap = nx.get_node_attributes(G_roadmap, "pos")

fig, axes = plt.subplots(figsize=(6, 6))
nx.draw(
    G_roadmap,
    pos_roadmap,
    with_labels=True,
    node_color=NODE_COLOR,
    node_size=350,
    edge_color="0.75",
    ax=axes,
)
axes.set_aspect("equal")

## Dijkstra and A* on the roadmap

Start and goal are the nodes closest to the lower left and the upper right corner.

Dijkstra uses the edge weights only. A* additionally uses a heuristic. For geometric graphs, the straight-line distance to the goal is a natural admissible heuristic. Both find a shortest path, but A* has to expand fewer nodes. To count the expanded nodes, the weight is given as a function: NetworkX calls it for every edge of a node that is expanded.

In [ ]:
def closest_node(graph, point):
    return min(graph.nodes, key=lambda node: math.dist(graph.nodes[node]["pos"], point))

start_node = closest_node(G_roadmap, (0, 0))
goal_node = closest_node(G_roadmap, (1, 1))

def euclidean_heuristic(node_a, node_b):
    pos_a = G_roadmap.nodes[node_a]["pos"]
    pos_b = G_roadmap.nodes[node_b]["pos"]
    return math.dist(pos_a, pos_b)

def counting_weight(expanded_nodes):
    """Edge weight as function, which additionally records the expanded nodes."""
    def weight(node_a, node_b, edge_data):
        expanded_nodes.add(node_a)
        return edge_data["weight"]
    return weight

expanded = {"Dijkstra": set(), "A*": set()}

if nx.has_path(G_roadmap, start_node, goal_node):
    dijkstra_path = nx.dijkstra_path(G_roadmap, start_node, goal_node, weight=counting_weight(expanded["Dijkstra"]))
    astar_path = nx.astar_path(
        G_roadmap,
        start_node,
        goal_node,
        heuristic=euclidean_heuristic,
        weight=counting_weight(expanded["A*"]),
    )
    print("Dijkstra:", dijkstra_path, "- expanded nodes:", len(expanded["Dijkstra"]))
    print("A*:", astar_path, "- expanded nodes:", len(expanded["A*"]))
    print("Path length:", nx.path_weight(G_roadmap, astar_path, weight="weight"))
else:
    print("No path found. Increase connection_radius or choose nodes in the same component.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

for ax, (name, path) in zip(axes, [("Dijkstra", dijkstra_path), ("A*", astar_path)]):
    node_colors = ["0.55" if node in expanded[name] else NODE_COLOR for node in G_roadmap]   # expanded: gray
    nx.draw(G_roadmap, pos_roadmap, ax=ax, with_labels=True, node_color=node_colors, node_size=350, edge_color="0.8")
    solution_edges = list(zip(path[:-1], path[1:]))
    nx.draw_networkx_edges(G_roadmap, pos_roadmap, ax=ax, edgelist=solution_edges, edge_color=PATH_COLOR, width=4)
    nx.draw_networkx_nodes(G_roadmap, pos_roadmap, ax=ax, nodelist=[start_node], node_color=START_COLOR, node_size=600)
    nx.draw_networkx_nodes(G_roadmap, pos_roadmap, ax=ax, nodelist=[goal_node], node_color=GOAL_COLOR, node_size=600)
    ax.set_title(f"{name}: {len(expanded[name])} expanded nodes (gray)")
    ax.set_aspect("equal")

## Handling disconnected graphs

Roadmaps are not always connected. Before searching a path, check whether start and goal lie in the same connected component. This avoids exceptions and makes planner failures easier to diagnose.

The same nodes as above, but with a smaller connection radius:

In [ ]:
G_sparse = build_roadmap(num_nodes, connection_radius=0.2, seed=7)   # same seed -> same node positions
components = list(nx.connected_components(G_sparse))

print("Component sizes:", sorted([len(component) for component in components], reverse=True))
print("Start and goal in same component:", any(start_node in component and goal_node in component for component in components))

try:
    path = nx.shortest_path(G_sparse, start_node, goal_node, weight="weight")
    print("Shortest path:", path)
except nx.NetworkXNoPath:
    print("NetworkXNoPath: start and goal are not connected.")

In [ ]:
start_component = nx.node_connected_component(G_sparse, start_node)
goal_component = nx.node_connected_component(G_sparse, goal_node)
node_colors = [START_COLOR if node in start_component else GOAL_COLOR if node in goal_component else NODE_COLOR
               for node in G_sparse]

fig, axes = plt.subplots(figsize=(6, 6))
nx.draw(G_sparse, pos_roadmap, ax=axes, with_labels=True, node_color=node_colors, node_size=350, edge_color="0.6")
axes.set_title("component of the start (green) and of the goal (red)")
axes.set_aspect("equal")

## Roadmap quality indicators

Simple graph metrics help to assess whether a sampled roadmap is likely to be useful for planning.

In [ ]:
largest_component = max(nx.connected_components(G_roadmap), key=len)
average_degree = sum(dict(G_roadmap.degree()).values()) / G_roadmap.number_of_nodes()
print("Nodes:", G_roadmap.number_of_nodes())
print("Edges:", G_roadmap.number_of_edges())
print("Average degree:", average_degree)
print("Density:", nx.density(G_roadmap))
print("Largest component size:", len(largest_component))

Conclusion
==========

Using and drawing of graphs can be done very nicely with the combination of networkX and matplotlib. The most important functions for the following notebooks:

| Task | NetworkX |
| :--- | :--- |
| create a graph | `nx.Graph()`, `nx.DiGraph()` |
| add nodes and edges with attributes | `G.add_node(n, pos=(x, y))`, `G.add_edge(u, v, weight=w)` |
| read attributes | `G.nodes[n]["pos"]`, `G[u][v]["weight"]`, `nx.get_node_attributes(G, "pos")` |
| neighbors and degree | `G.neighbors(n)`, `G.degree(n)` |
| connectivity | `nx.has_path(G, s, g)`, `nx.connected_components(G)`, `nx.is_connected(G)` |
| shortest path | `nx.shortest_path(G, s, g, weight="weight")`, `nx.astar_path(G, s, g, heuristic=h, weight="weight")` |
| costs of a path | `nx.path_weight(G, path, weight="weight")`, `nx.shortest_path_length(G, s, g, weight="weight")` |
| drawing | `nx.draw(G, pos, with_labels=True)`, `nx.draw_networkx_edges(G, pos, edgelist=...)`, `nx.draw_networkx_edge_labels(G, pos, edge_labels=...)` |

# Exercise


1. Create a graph with `x` nodes at random positions.
2. Create a graph with `x` nodes at random positions and connect all nodes with edges.
3. Create a graph with `x` nodes at random positions and connect nodes whose distance is below a given threshold.
4. Store the Euclidean distance as edge weight.
5. Choose start and goal nodes and find the weighted shortest path.
6. Draw the graph, mark start and goal, and highlight the solution path.
7. Change the distance threshold and observe how connectivity, average degree, and path length change.
